# Streaming

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangGraph roadmap.

You will learn the 7 LangGraph stream modes, token-by-token output with `messages` mode, and custom streaming with `get_stream_writer()`.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Build a Simple Chat Graph

Create a basic chatbot graph to demonstrate streaming.

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

def chatbot(state: MessagesState) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}

graph = StateGraph(MessagesState)
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

app = graph.compile()

## 4. Stream Mode: values

Emits the full state after each node completes.

In [ ]:
for state in app.stream({"messages": [("human", "What is LangGraph?")]}, stream_mode="values"):
    if state["messages"]:
        last = state["messages"][-1]
        print(f"[{last.type}] {last.content[:100]}..." if len(last.content) > 100 else f"[{last.type}] {last.content}")

## 5. Stream Mode: updates

Emits only the state delta returned by each node.

In [ ]:
for update in app.stream({"messages": [("human", "What is LangGraph?")]}, stream_mode="updates"):
    for node_name, node_output in update.items():
        print(f"Node '{node_name}' returned {len(node_output.get('messages', []))} message(s)")

## 6. Stream Mode: messages (Token-by-Token)

Streams individual LLM tokens as they are generated.

In [ ]:
for chunk, metadata in app.stream(
    {"messages": [("human", "Write a haiku about coding")]},
    stream_mode="messages",
):
    if chunk.content:
        print(chunk.content, end="", flush=True)
print()

## 7. Custom Streaming with get_stream_writer()

Send arbitrary status updates from inside a node using `get_stream_writer()`.

In [ ]:
from langgraph.config import get_stream_writer
import time

def research_node(state: MessagesState) -> dict:
    writer = get_stream_writer()
    writer({"status": "researching", "progress": 0})
    time.sleep(1)
    writer({"status": "researching", "progress": 50})
    response = llm.invoke(state["messages"])
    writer({"status": "complete", "progress": 100})
    return {"messages": [response]}

graph2 = StateGraph(MessagesState)
graph2.add_node("research", research_node)
graph2.add_edge(START, "research")
graph2.add_edge("research", END)
app2 = graph2.compile()

for event in app2.stream(
    {"messages": [("human", "Explain quantum computing briefly")]},
    stream_mode="custom",
):
    print(f"Custom event: {event}")

## 8. Combining Multiple Stream Modes

Pass a list of modes and use `version="v2"` for consistent event formatting.

In [ ]:
for event in app.stream(
    {"messages": [("human", "Explain recursion in one sentence")]},
    stream_mode=["updates", "messages"],
    version="v2",
):
    if event.mode == "messages":
        chunk, meta = event.data
        if chunk.content:
            print(chunk.content, end="", flush=True)
    elif event.mode == "updates":
        print(f"\n[update] Node finished: {list(event.data.keys())}")
print()

## Stream Modes Reference

| Mode | Description |
|------|-------------|
| `values` | Full state after each node |
| `updates` | State delta from each node |
| `messages` | LLM tokens as `(chunk, metadata)` |
| `custom` | Custom data via `get_stream_writer()` |
| `checkpoints` | Checkpoint events after each step |
| `tasks` | Task start/end events |
| `debug` | Detailed debug events |

## What You Learned

1. LangGraph supports **7 stream modes** for different use cases
2. **`messages` mode** streams LLM tokens for real-time chat output
3. **`get_stream_writer()`** lets you emit custom progress events from nodes
4. **Multiple modes** can be combined with `version="v2"` for structured events
5. Streaming makes long-running graphs feel responsive to end users